In [0]:
# Standard Batch Read (Non-streaming)
raw_df = spark.read.option("multiLine", "true").json("/Volumes/main/lufthansa/landing_zone/ops/flights/*/*.json")

# Add a simple ingestion timestamp
from pyspark.sql.functions import current_timestamp
bronze_df = raw_df.withColumn("bronze_at", current_timestamp())

# Overwrite or Append to the table
bronze_df.write.format("delta").mode("overwrite").save("/Volumes/main/lufthansa/bronze/flights")

In [0]:
# Point Spark at the folder, not the parquet file
df = spark.read.format("delta").load("/Volumes/main/lufthansa/bronze/flights")

# This will show you the columns: ingestion_metadata, payload, etc.
display(df)

In [0]:
%sql
-- This treats the folder as a table
SELECT 
  payload.FlightStatusResource.Flights.Flight[0].OperatingCarrier.FlightNumber as first_flight,
  ingestion_metadata.ingested_at
FROM delta.`/Volumes/main/lufthansa/bronze/flights`

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS main.lufthansa._checkpoints;

In [0]:

dbutils.fs.mkdirs("/Volumes/main/lufthansa/_checkpoints/bronze_flights")
dbutils.fs.mkdirs("/Volumes/main/lufthansa/_checkpoints/bronze_flights/schema")
dbutils.fs.mkdirs("/Volumes/main/lufthansa/_checkpoints/bronze_flights/data")

In [0]:
from pyspark.sql.functions import col, current_timestamp, input_file_name

# Configuration
source_base_path = "/Volumes/main/lufthansa/landing_zone/ref/"
checkpoint_base = "/Volumes/main/lufthansa/_checkpoints/bronze_ref/"
target_catalog = "main"
target_schema = "lufthansa_bronze"

def ingest_reference_data(ref_type):
    source_path = f"{source_base_path}{ref_type}/*"
    checkpoint_path = f"{checkpoint_base}{ref_type}"
    target_table = f"{target_catalog}.{target_schema}.ref_{ref_type}"
    
    print(f"Starting ingestion for: {ref_type}")

    # Auto Loader Stream
    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        # Automatically infer schema and handle changes
        .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
        .option("cloudFiles.schemaEvolutionMode", "rescue") 
        .option("cloudFiles.inferColumnTypes", "true")
        .load(source_path)
        # Add crucial Bronze metadata
        .withColumn("_input_file", input_file_name())
        .withColumn("_ingested_at", current_timestamp())
        .writeStream
        .format("delta")
        .option("checkpointLocation", f"{checkpoint_path}/data")
        .option("mergeSchema", "true")
        .trigger(availableNow=True) # "Batch-style" streaming
        .toTable(target_table))

# List your reference types to process
ref_types = ["airports", "countries", "cities", "airlines"]

for ref in ref_types:
    ingest_reference_data(ref)